**Meyiapkan Dataset**

In [13]:
import numpy as np
import pandas as pd

np.random.seed(111)
n = 2500

menu_list = [
    "Kopi Susu", "Americano", "Cappuccino",
    "Roti Bakar", "Kentang Goreng", "Es Teh"
]

cabang_list = ["Magelang", "Yogyakarta", "Semarang"]
waktu_list = ["Pagi", "Siang", "Sore", "Malam"]

data_tugas11 = pd.DataFrame({
    "nama_menu": np.random.choice(menu_list, size=n),
    "cabang": np.random.choice(cabang_list, size=n),
    "waktu_pesan": np.random.choice(waktu_list, size=n),
    "qty": np.random.randint(1, 6, size=n),
    "harga_satuan": np.random.choice(
        [15000, 18000, 20000, 22000, 25000, 30000],
        size=n
    )
})

data_tugas11["total_penjualan"] = (
    data_tugas11["qty"] * data_tugas11["harga_satuan"]
    + np.random.normal(0, 8000, size=n)
)

data_tugas11.to_csv("tugas11_penjualan.csv", index=False)

print("Dataset berhasil dibuat:", len(data_tugas11), "baris")
print(data_tugas11.head())

Dataset berhasil dibuat: 2500 baris
        nama_menu      cabang waktu_pesan  qty  harga_satuan  total_penjualan
0  Kentang Goreng    Semarang       Malam    1         30000     27750.163532
1  Kentang Goreng  Yogyakarta        Sore    3         20000     64216.160744
2  Kentang Goreng    Semarang       Siang    5         22000    108722.792592
3  Kentang Goreng  Yogyakarta        Pagi    2         20000     54483.205872
4      Roti Bakar    Magelang       Malam    2         30000     58157.944120


In [ ]:
**Nyalakan SparkSession

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder \
    .appName("Tugas11-KopiNusantara") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

# Membaca dataset
df_tugas = spark.read.csv(
    "tugas11_penjualan.csv",
    header=True,
    inferSchema=True
)

df_tugas.show(5)
print("Jumlah data:", df_tugas.count())

+--------------+----------+-----------+---+------------+------------------+
|     nama_menu|    cabang|waktu_pesan|qty|harga_satuan|   total_penjualan|
+--------------+----------+-----------+---+------------+------------------+
|Kentang Goreng|  Semarang|      Malam|  1|       30000|27750.163531558544|
|Kentang Goreng|Yogyakarta|       Sore|  3|       20000| 64216.16074371864|
|Kentang Goreng|  Semarang|      Siang|  5|       22000|108722.79259154423|
|Kentang Goreng|Yogyakarta|       Pagi|  2|       20000|54483.205871792496|
|    Roti Bakar|  Magelang|      Malam|  2|       30000| 58157.94412015849|
+--------------+----------+-----------+---+------------+------------------+
only showing top 5 rows

Jumlah data: 2500


**A. Feature Engineering** *(bobot 20%)*

Gunakan `StringIndexer` untuk mengubah kolom `nama_menu`, `cabang`, dan `waktu_pesan` menjadi indeks numerik. Gabungkan seluruh fitur (`qty`, `harga_satuan`, dan ketiga kolom hasil indexing) menjadi satu kolom `features` menggunakan `VectorAssembler`.

In [5]:
from pyspark.ml.feature import StringIndexer, VectorAssembler

# StringIndexer untuk nama menu
indexer_menu = StringIndexer(
    inputCol="nama_menu",
    outputCol="nama_menu_idx")

# StringIndexer untuk cabang
indexer_cabang = StringIndexer(
    inputCol="cabang",
    outputCol="cabang_idx")

# StringIndexer untuk waktu pesan
indexer_waktu = StringIndexer(
    inputCol="waktu_pesan",
    outputCol="waktu_pesan_idx")

# Melatih dan menerapkan StringIndexer
model_indexer_menu = indexer_menu.fit(df_tugas)
df_indexed = model_indexer_menu.transform(df_tugas)

model_indexer_cabang = indexer_cabang.fit(df_indexed)
df_indexed = model_indexer_cabang.transform(df_indexed)

model_indexer_waktu = indexer_waktu.fit(df_indexed)
df_indexed = model_indexer_waktu.transform(df_indexed)

# Menggabungkan seluruh fitur
assembler = VectorAssembler(
    inputCols=["qty", "harga_satuan", "nama_menu_idx", "cabang_idx", "waktu_pesan_idx"],
    outputCol="features")

df_final = assembler.transform(df_indexed)
df_final.select("nama_menu", "cabang", "waktu_pesan", "features").show(5, truncate=False)

+--------------+----------+-----------+-------------------------+
|nama_menu     |cabang    |waktu_pesan|features                 |
+--------------+----------+-----------+-------------------------+
|Kentang Goreng|Semarang  |Malam      |[1.0,30000.0,5.0,2.0,1.0]|
|Kentang Goreng|Yogyakarta|Sore       |[3.0,20000.0,5.0,1.0,3.0]|
|Kentang Goreng|Semarang  |Siang      |[5.0,22000.0,5.0,2.0,2.0]|
|Kentang Goreng|Yogyakarta|Pagi       |[2.0,20000.0,5.0,1.0,0.0]|
|Roti Bakar    |Magelang  |Malam      |[2.0,30000.0,4.0,0.0,1.0]|
+--------------+----------+-----------+-------------------------+
only showing top 5 rows



**B. Train/Test Split & Pelatihan Model** *(bobot 20%)*

Bagi data menjadi 75% training dan 25% testing (`seed=101`). Latih model `LinearRegression` dengan `labelCol="total_penjualan"`.


In [6]:
from pyspark.ml.regression import LinearRegression

# Membagi data
train, test = df_final.randomSplit([0.75, 0.25], seed=101)

print("Jumlah data training:", train.count())
print("Jumlah data testing:", test.count())

# Membuat model Linear Regression
lr = LinearRegression(
    featuresCol="features",
    labelCol="total_penjualan")

# Melatih model
model_utama = lr.fit(train)

print("Model berhasil dilatih!")
print("Koefisien:", model_utama.coefficients)
print("Intercept:", model_utama.intercept)

Jumlah data training: 1839
Jumlah data testing: 661
Model berhasil dilatih!
Koefisien: [22046.663271025755,3.0229208681606696,-123.21425598657828,279.2845246167609,339.63879532317213]
Intercept: -66903.1776648588


**C. Evaluasi Model** *(bobot 20%)*

Hitung dan tampilkan **RMSE** dan **R²** dari model pada data testing. Tulis 1-2 kalimat interpretasi: apakah model ini cukup baik untuk dipakai memprediksi omzet?


In [7]:
from pyspark.ml.evaluation import RegressionEvaluator

# Melakukan prediksi pada data testing
predictions = model_utama.transform(test)

# Menghitung RMSE
evaluator_rmse = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="rmse")

rmse = evaluator_rmse.evaluate(predictions)

# Menghitung R²
evaluator_r2 = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="r2")

r2 = evaluator_r2.evaluate(predictions)

print(f"RMSE: Rp {rmse:,.0f}")
print(f"R²: {r2:.4f}")

RMSE: Rp 10,901
R²: 0.9107


**D. Eksperimen Perbandingan Fitur** *(bobot 25%)*

Latih **model kedua** yang HANYA menggunakan fitur `qty` dan `harga_satuan` (tanpa informasi menu/cabang/waktu). Bandingkan R² model kedua ini dengan model utama pada bagian C. Tulis kesimpulan: apakah menambahkan informasi menu/cabang/waktu benar-benar meningkatkan akurasi prediksi secara berarti?

In [8]:
# Menggabungkan dua fitur saja
assembler_dua = VectorAssembler(
    inputCols=["qty", "harga_satuan"],
    outputCol="features_dua")
df_dua = assembler_dua.transform(df_tugas)

# Membagi data dengan seed yang sama
train_dua, test_dua = df_dua.randomSplit([0.75, 0.25], seed=101)

# Membuat model kedua
lr_dua = LinearRegression(featuresCol="features_dua", labelCol="total_penjualan")
model_dua = lr_dua.fit(train_dua)

# Melakukan prediksi
predictions_dua = model_dua.transform(test_dua)

# Menghitung R² model kedua
evaluator_dua = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="r2")
r2_dua = evaluator_dua.evaluate(predictions_dua)

# Membandingkan kedua model
print(f"R² Model Utama (5 fitur): {r2:.4f}")
print(f"R² Model Kedua (2 fitur): {r2_dua:.4f}")

print("Selisih R²:", round(r2 - r2_dua, 4))

if r2 > r2_dua:
    print("Model utama memiliki akurasi lebih baik.")
elif r2 < r2_dua:
    print("Model kedua memiliki akurasi lebih baik.")
else:
    print("Kedua model memiliki akurasi yang sama.")

R² Model Utama (5 fitur): 0.9107
R² Model Kedua (2 fitur): 0.9108
Selisih R²: -0.0001
Model kedua memiliki akurasi lebih baik.


**E. Prediksi Skenario Baru** *(bobot 15%)*

Buat sebuah DataFrame baru berisi **3 skenario pesanan (data dummy)** (boleh bebas kombinasi menu/cabang/waktu/qty/harga, namun harus melalui tahapan `StringIndexer` dan `VectorAssembler` yang sama seperti data training — perhatikan: gunakan model `StringIndexer` yang **sama** yang sudah di-`fit()` sebelumnya, jangan membuat `StringIndexer` baru). Tampilkan hasil prediksi `total_penjualan` untuk ketiga skenario tersebut.

In [10]:
# Membuat DataFrame berisi 3 skenario pesanan baru
data_baru = spark.createDataFrame([
    ("Kopi Susu", "Magelang", "Pagi", 2, 18000),
    ("Cappuccino", "Yogyakarta", "Siang", 3, 25000),
    ("Roti Bakar", "Semarang", "Malam", 1, 22000)],
    ["nama_menu", "cabang", "waktu_pesan", "qty", "harga_satuan"])

# Menggunakan StringIndexer yang sudah dilatih sebelumnya
data_baru = model_indexer_menu.transform(data_baru)
data_baru = model_indexer_cabang.transform(data_baru)
data_baru = model_indexer_waktu.transform(data_baru)

# Menggabungkan fitur menggunakan VectorAssembler yang sama
data_baru = assembler.transform(data_baru)

# Melakukan prediksi menggunakan model utama
hasil_prediksi = model_utama.transform(data_baru)

# Menampilkan hasil prediksi
hasil_prediksi.select("nama_menu", "cabang", "waktu_pesan", "qty", "harga_satuan", "prediction").show(3, truncate=False)

+----------+----------+-----------+---+------------+------------------+
|nama_menu |cabang    |waktu_pesan|qty|harga_satuan|prediction        |
+----------+----------+-----------+---+------------+------------------+
|Kopi Susu |Magelang  |Pagi       |2  |18000       |31479.510248098188|
|Cappuccino|Yogyakarta|Siang      |3  |25000       |75521.96745552514 |
|Roti Bakar|Semarang  |Malam      |1  |22000       |22053.09552631207 |
+----------+----------+-----------+---+------------+------------------+



**Explorasi**

In [11]:
from pyspark.sql.functions import col, abs, desc
import matplotlib.pyplot as plt

# Membuat kolom selisih prediksi dan nilai aktual
hasil_residual = predictions.withColumn("residual", col("total_penjualan") - col("prediction"))

# Menghitung nilai absolut kesalahan
hasil_residual = hasil_residual.withColumn("error_absolut", abs(col("residual")))

# Menampilkan 10 data dengan kesalahan terbesar
print("10 Prediksi dengan Kesalahan Terbesar:")

hasil_residual.orderBy(desc("error_absolut")).select("qty", "harga_satuan", "total_penjualan", "prediction", "residual", "error_absolut").show(10, truncate=False)

# Mengubah hasil residual menjadi Pandas
df_residual = hasil_residual.select("total_penjualan", "prediction", "residual").toPandas()

# Visualisasi residual
plt.figure(figsize=(9, 5))

plt.scatter(df_residual["prediction"], df_residual["residual"], alpha=0.5)

# Garis nol sebagai acuan
plt.axhline(y=0, linestyle="--")

plt.title("Residual Analysis - Linear Regression")
plt.xlabel("Hasil Prediksi (Rp)")
plt.ylabel("Residual (Rp)")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Menghitung rata-rata kesalahan absolut
mae = df_residual["residual"].abs().mean()

print(f"Rata-rata Kesalahan Absolut (MAE): Rp {mae:,.2f}")

10 Prediksi dengan Kesalahan Terbesar:
+---+------------+------------------+------------------+-------------------+------------------+
|qty|harga_satuan|total_penjualan   |prediction        |residual           |error_absolut     |
+---+------------+------------------+------------------+-------------------+------------------+
|5  |15000       |51000.1029958954  |89788.58409657328 |-38788.48110067788 |38788.48110067788 |
|1  |15000       |33555.768174980054|150.51187326086801|33405.256301719186 |33405.256301719186|
|1  |15000       |33705.338400059096|676.2249098507891 |33029.11349020831  |33029.11349020831 |
|5  |15000       |56699.912169753916|89292.87503261994 |-32592.96286286602 |32592.96286286602 |
|5  |15000       |56882.30224973163 |88181.0946887337  |-31298.792439002078|31298.792439002078|
|5  |30000       |164555.4673424478 |134976.32685035316|29579.14049209465  |29579.14049209465 |
|5  |30000       |163957.68151664536|134790.25260908646|29167.428907558904 |29167.428907558904|
|

<Figure size 900x500 with 1 Axes>

Rata-rata Kesalahan Absolut (MAE): Rp 8,733.63


In [12]:
spark.stop
print("SparkSEssion ditutup.")

SparkSEssion ditutup.
